In [1]:
import torch 
from reasoning_from_scratch.ch02 import get_device 
from reasoning_from_scratch.ch03 import (
    load_model_and_tokenizer 
)

device = get_device()
device = torch.device('cpu')

model, tokenizer = load_model_and_tokenizer(
    which_model = 'base',
    device = device,
    use_compile = False 
)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [2]:
from reasoning_from_scratch.ch03 import render_prompt 
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache 
)
raw_prompt = (
    'Half the value of $3x-9$ is $x+37$. ' 
    'What is the value of $x$?' 
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device, 
    max_new_tokens = 2048, verbose = True, 
    generate_func = generate_text_top_p_stream_cache,
    temperature = 0.9,
    top_p = 0.9
)
print(response)

 \boxed{58} \boxed{58}


In [3]:
import json 
import requests 
from pathlib import Path 

def load_math_train(local_path = 'math_train.json', save_copy = True):
    local_path = Path(local_path)
    url = (
        'https://raw.githubusercontent.com/rabst/' 
        'math_full_minus_math500/refs/heads/main/' 
        'math_full_minus_math500.json' 
    )
    backup_url = (
        'https://f001.backblazeb2.com/file/reasoning-from-scratch/' 
        'MATH/math_full_minus_math500.json' 
    )
    
    if local_path.exists():
        with local_path.open('r', encoding = 'utf-8') as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout = 30)
            r.raise_for_status()
        except requests.RequestException:
            print('using backup URL')
            r = requests.get(backup_url, timeout = 30)
            r.raise_for_status()
            
        data = r.json()
        
        if save_copy:
            with local_path.open('w', encoding = 'utf-8') as f:
                json.dump(data, f, indent = 2)
    
    return data
    
math_train = load_math_train()
print('dataset size:', len(math_train))

dataset size: 12000


In [4]:
from pprint import pprint 
pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

In [5]:
from reasoning_from_scratch.qwen3 import KVCache 
from reasoning_from_scratch.ch04 import top_p_filter

@torch.no_grad()

def sample_response(
    model, 
    tokenizer,
    prompt, device,
    max_new_tokens = 512,
    temperature = 0.8,
    top_p = 0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device = device 
    )
    
    cache = KVCache(n_layers = model.cfg['n_layers'])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache = cache)[:, -1]
    
    generated = []
    for _ in range(max_new_tokens):
        if temperature is not None and temperature != 0 and temperature != 1:
            logits = logits / temperature
        
        probas = torch.softmax(logits, dim = -1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples = 1
        ).to(device)
        
        token_id = next_token.item()
        generated.append(token_id)
        
        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break 
        
        logits = model(next_token, cache = cache)[:, -1]
            
    full_token_ids = torch.cat([input_ids, torch.tensor(generated, device = device, dtype = input_ids.dtype),])
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [6]:
torch.manual_seed(5)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
    model = model,
    tokenizer = tokenizer,
    prompt = prompt,
    device = device,
    max_new_tokens = 512,
    temperature = 0.9,
    top_p = 0.9,
)
print(repr(answer_text))

" Let's solve the problem step by step.\n\n**Given:**\n\\[\n\\text{Half the value of } 3x - 9 \\text{ is } x + 37.\n\\]\n\n**Step 1: Translate the statement into an equation.**\n\\[\n\\frac{1}{2} (3x - 9) = x + 37\n\\]\n\n**Step 2: Eliminate the fraction by multiplying both sides by 2.**\n\\[\n3x - 9 = 2(x + 37)\n\\]\n\n**Step 3: Distribute the 2 on the right side.**\n\\[\n3x - 9 = 2x + 74\n\\]\n\n**Step 4: Subtract \\(2x\\) from both sides to get the \\(x\\)-terms on one side.**\n\\[\n3x - 2x - 9 = 74\n\\]\n\\[\nx - 9 = 74\n\\]\n\n**Step 5: Add 9 to both sides to solve for \\(x\\).**\n\\[\nx = 74 + 9\n\\]\n\\[\nx = 83\n\\]\n\n**Final Answer:**\n\\[\n\\boxed{83}\n\\]<|endoftext|>"


In [7]:
rollouts = [
r"\boxed{83}",
r"The correct answer is \boxed{83}",
r"The final answer is 83",
r"We get \boxed{38}",
]

In [21]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer 
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback = None 
    )
    if not extracted:
        return 0.0 
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [22]:
rollout_rewards = []

for answer in rollouts:
    reward = reward_rlve(answer_text = answer, ground_truth = '83')
    print(f'answer: {answer!r}')
    print(f'reward: {reward}\n')
    rollout_rewards.append(reward)

answer: '\\boxed{83}'
reward: 1.0

answer: 'The correct answer is \\boxed{83}'
reward: 1.0

answer: 'The final answer is 83'
reward: 0.0

answer: 'We get \\boxed{38}'
reward: 0.0



In [23]:
rewards = torch.tensor(rollout_rewards, device = device)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
print(rewards)
print(advantages)

tensor([1., 1., 0., 0.])
tensor([ 0.8659,  0.8659, -0.8659, -0.8659])


In [24]:
@torch.inference_mode()

def avg_logprob_answer(model, tokenizer, prompt, answer, device = 'cpu'):
    
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device = device)
    
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim = -1)
    
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1
    
    t_idx = torch.arange(start, end, device = device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]
    
    return torch.mean(next_token_logps).item()

avg_logprob_val = avg_logprob_answer(
    model, tokenizer,
    prompt = prompt,
    answer = answer_text,
    device = device 
)
print(avg_logprob_val)

-0.061279296875


In [25]:
sequence_logprob_val = avg_logprob_val * (
    len(tokenizer.encode(answer_text))
)
print(sequence_logprob_val)

-16.239013671875


In [26]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim = -1)
    
    start = prompt_len -1
    end = token_ids.shape[0] - 1
    
    t_idx = torch.arange(start, end, device = token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]
    
    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

AttributeError: 'list' object has no attribute 'unsqueeze'

In [ ]:
def sequence_logprob(model, token_ids, prompt_len):
    
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim = -1)
    selected = logprobs[:-1].gather(1, token_ids[1:].unsqueeze(-1)).squeeze(-1)     # tensor.gather(dim, index)
    
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

tensor(-16.2998, grad_fn=<SumBackward0>)


In [ ]:
rollouts = [
r"\boxed{83}",
r"The correct answer is \boxed{83}",
r"The final answer is 83",
r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + ' ' + text)
    logprob = sequence_logprob(
        model = model,
        token_ids = torch.tensor(token_ids, device = device),
        prompt_len = prompt_len,
)
    print(f'answer: {text}')
    print(f'longprob: {logprob.item():.4f}\n')

    rollout_logps.append(logprob)

answer: \boxed{83}
longprob: -7.9243

answer: The correct answer is \boxed{83}
longprob: -20.1546

answer: The final answer is 83
longprob: -16.6130

answer: We get \boxed{38}
longprob: -23.3677



In [ ]:
logps = torch.stack(rollout_logps)
pg_loss = -(advantages.detach() * logps).mean()

print(logps)
print(pg_loss)

tensor([ -7.9243, -20.1546, -16.6130, -23.3677], grad_fn=<StackBackward0>)
tensor(-2.5764, grad_fn=<NegBackward0>)


In [27]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts = 2,
    max_new_tokens = 256,
    temperature = 0.8,
    top_p = 0.9
):
    
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example['problem'])
    
    was_training = model.training 
    model.eval()
    
    for _ in range(num_rollouts):
        token_ids, prompt_len, text = sample_response(
            model = model,
            tokenizer = tokenizer,
            prompt = prompt,
            device = device,
            max_new_tokens = max_new_tokens,
            temperature = temperature,
            top_p = top_p,
        )
        
        reward = reward_rlvr(text, example['answer'])
        logp = sequence_logprob(model, token_ids, prompt_len)
        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                'text': text,
                'reward': reward,
                'gen_len': token_ids.numel() - prompt_len,
            }
        )
    if was_training:
        model.train()
    
    rewards = torch.tensor(roll_rewards, device = device)
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
    logps = torch.stack(roll_logps)
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss 
    
    return {
        'loss': loss.item(),
        'pg_loss': pg_loss.item(),
        'rewards': roll_rewards,
        'advantages': advantages.detach().cpu().tolist(),
        'samples': samples,
        'loss_tensor': loss,
    }

In [28]:
torch.manual_seed(123)

stats = compute_grpo_loss(
    model = model,
    tokenizer = tokenizer,
    example = math_train[4],
    device = device,
    num_rollouts = 2,
    max_new_tokens = 256,
    temperature = 0.8,\
    top_p = 0.9
)
print(stats)

{'loss': -0.0, 'pg_loss': -0.0, 'rewards': [0.0, 0.0], 'advantages': [0.0, 0.0], 'samples': [{'text': ' 14<|endoftext|>', 'reward': 0.0, 'gen_len': 4}, {'text': " 4\n\nTo solve the problem, let's break it down step by step:\n\n1. **Define Variables:**\n   - Let \\( x \\) be the number of days Sam works.\n   - Then, the number of days he does not work is \\( 20 - x \\).\n\n2. **Set Up the Earnings Equation:**\n   - For each day he works, he earns \\$60.\n   - For each day he does not work, he loses \\$30.\n   - His total earnings are \\$660.\n\n   The equation is:\n   \\[\n   60x - 30(20 - x) = 660\n   \\]\n\n3. **Simplify the Equation:**\n   \\[\n   60x - 600 + 30x = 660\n   \\]\n   \\[\n   90x - 600 = 660\n   \\]\n\n4. **Solve for \\( x \\):**\n   \\[\n   90x = 660 + 600\n   \\]\n   \\[\n   90x = 1260\n   \\]\n   \\[\n", 'reward': 0.0, 'gen_len': 256}], 'loss_tensor': tensor(-0., grad_fn=<NegBackward0>)}


In [ ]:
import time 

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps = None,
    num_rollouts = 2,
    max_new_tokens = 256,
    temperature = 0.8,
    top_p = 0.9,
    lr = 1e-5,
    checkpoint_every = 50,
    checkpoint_dir = '.',
    csv_log_path = None,
):
    if steps is None:
        steps = len(math_data)
        
    optimizer = torch.optim.AdamW(model.parameters(), lr = lr)

    model.train()
    current_step = 0
    
    if csv_log_path is None:
        timestamp = time.strftime('%Y%m%d_%H%M%S')                  # strftime means string format time
        csv_log_path = f'train_rlvr_grpo_metrics{timestamp}.csv' 
    csv_log_path = Path(csv_log_path)
    
    try:
        for step in range(steps):
            
            optimizer.zero_grad()
            
            current_step = step + 1
            example = math_data[step % len(math_data)]
            
            stats = compute_grpo_loss(
                model = model,
                tokenizer = tokenizer,
                example = example,
                device = device, 
                num_rollouts = num_rollouts,
                max_new_tokens = max_new_tokens,
                temperature = temperature,
                top_p = top_p,
            )
            
            stats['loss_tensor'].backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            
            reward_avg = torch.tensor(stats['rewards']).mean().item()
            step_tokens = sum(
                sample['gen_len'] for sample in stats['samples']
            )
            
            avg_response_len = (
                step_tokens / len(stats['samples']) if stats['samples'] else 0.0
            )
            
            append_csv_metrics(
                csv_log_path, current_step, steps,
                stats['loss'], reward_avg, avg_response_len,
            )
            
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )
            
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model = model,
                    checkpoint_dir = checkpoint_dir,
                    step = current_step,
                )
                print(f'saved checkpoint to {ckpt_path}')
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model = model,
            checkpoint_dir = checkpoint_dir,
            step = max(1, current_step),
            suffix = 'interrupt',
        )
        print(f'\nKeyboardInterrupt. saved checkpoint to {ckpt_path}')
        return model 
    
    return model 

def save_checkpoint(model, checkpoint_dir, step, suffix = ''):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents = True, exist_ok = True)
    suffix = f'-{suffix}' if suffix else '' 
    ckpt_path = (
        checkpoint_dir / 
        f'qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth' 
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path 

def append_csv_metrics(
    csv_log_path, step_idx, total_steps,
    loss, reward_avg, avg_response_len,
):
    
    if not csv_log_path.exists():
        csv_log_path.write_text(
            'steps,total_steps,loss,reward_avg,avg_response_len\n',
            encoding = 'utf-8',
        )
        
    with csv_log_path.open('a', encoding = 'utf-8') as f:
        f.write(
            f'{step_idx}, {total_steps}, {loss:.6f},{reward_avg:.6f},' 
            f'{avg_response_len:.6f}\n'
        )

In [ ]:
device = get_device()
model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model = model,
    tokenizer = tokenizer,
    math_data = math_train,
    device = device,
    steps = 50,
    num_rollouts = 4,
    max_new_tokens = 512,
    temperature = 0.8,
    top_p = 0.9,
    lr = 1e-5,
    checkpoint_every = 5,
    checkpoint_dir = '.',
)

Using Apple Silicon GPU (MPS)
[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.2
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.8
[Step 3/50] loss=9.1913 reward_avg=0.500 avg_resp_len=56.0
[Step 4/50] loss=2.9958 reward_avg=0.250 avg_resp_len=56.5
[Step 5/50] loss=14.9828 reward_avg=0.500 avg_resp_len=186.2
saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.pth

KeyboardInterrupt. saved checkpoint to qwen3-0.6B-rlvr-grpo-step00006-interrupt.pth


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [37]:
model.load_state_dict(
    torch.load("qwen3-0.6B-rlvr-grpo-step00005.pth")
)

<All keys matched successfully>

In [39]:
from reasoning_from_scratch.qwen3 import download_qwen3_grpo_checkpoints
download_qwen3_grpo_checkpoints(grpo_type="no_kl", step="00050")

qwen3-0.6B-rlvr-grpo-step00050.pth: 100% (1433 MiB / 1433 MiB)


PosixPath('qwen3-0.6B-rlvr-grpo-step00050.pth')